In [ ]:
# =============================================================================
# 02_derived_products.ipynb  —  HPC version
# Compute PET, D, SPEI (3/6/12), FVC, forest mask, and forest density.
#
# Inputs : /scratch/lustre/users/fngari/John/SOIL-MOISTURE-PREDICTION/data/processed/
# Outputs: .../data/derived/         (NetCDF stacks, all years)
#          .../data/derived_tifs/    (GeoTIFF snapshots, 6 years)
# =============================================================================

import os, sys, json, traceback
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from numba import njit, prange

# ---------------- Paths (HPC, hardcoded) ----------------
REPO_ROOT   = Path('/scratch/lustre/users/fngari/John/SOIL-MOISTURE-PREDICTION')
PROC_DIR    = REPO_ROOT / 'data' / 'processed'
OBJ1_DIR    = REPO_ROOT / 'data' / 'obj1'
DERIVED_DIR = REPO_ROOT / 'data' / 'derived'
TIF_DIR     = REPO_ROOT / 'data' / 'derived_tifs'
LOG_DIR     = REPO_ROOT / 'notebooks' / 'logs'

for p in [DERIVED_DIR, TIF_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ---------------- Config ----------------
SEASONS       = ['JF', 'MAM', 'JJAS', 'OND']
YEARS         = list(range(1995, 2026))
CRS           = 'EPSG:21037'
NODATA        = -9999

# Month-count weights for seasonal → annual
SEASON_MONTHS = {'JF': 2, 'MAM': 3, 'JJAS': 4, 'OND': 3}
TOTAL_MONTHS  = sum(SEASON_MONTHS.values())  # 12

# AOI centroid latitude for extraterrestrial radiation
LAT_DEG       = -0.15

# SPEI scales in season-steps (1 = 3-month, 2 = 6-month, 4 = 12-month)
SPEI_SCALES   = {'SPEI_3': 1, 'SPEI_6': 2, 'SPEI_12': 4}

# Snapshot years for GeoTIFF exports
SNAPSHOT_YEARS = [1995, 2001, 2007, 2013, 2019, 2025]

# Variables to export as TIFs (annual weighted mean across seasons)
TIF_VARIABLES  = ['PET', 'D', 'SPEI_3', 'SPEI_6', 'SPEI_12',
                  'FVC', 'FOREST_DENSITY']

# ---------------- Sanity ----------------
assert REPO_ROOT.exists(), f"Missing: {REPO_ROOT}"
assert PROC_DIR.exists(),  f"Missing: {PROC_DIR} — run Notebook 1 first"
assert (OBJ1_DIR / 'FOREST_MASK.tif').exists(), \
    f"Missing: {OBJ1_DIR / 'FOREST_MASK.tif'} — upload the forest mask"

print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"PROC_DIR    : {PROC_DIR}")
print(f"DERIVED_DIR : {DERIVED_DIR}")
print(f"TIF_DIR     : {TIF_DIR}")
print(f"Snapshot yrs: {SNAPSHOT_YEARS}")
print(f"TIF vars    : {TIF_VARIABLES}")

In [ ]:
# =============================================================================
# Load all inputs from Notebook 1.
# Handles variable naming inconsistencies (SRTM was saved with a default
# xarray variable name). Reports RSS after loads.
# =============================================================================

import gc, os, psutil

def load_da(path, var=None):
    ds = xr.open_dataset(path)
    if var is None:
        candidates = [k for k in ds.data_vars
                      if k not in ('x', 'y', 'spatial_ref', 'band')]
        if not candidates:
            raise RuntimeError(f"No data variable in {path.name}")
        return ds[candidates[0]]
    if var not in ds.data_vars:
        candidates = [k for k in ds.data_vars
                      if k not in ('x', 'y', 'spatial_ref', 'band')]
        print(f"⚠ {path.name}: '{var}' not found, using '{candidates[0]}'")
        return ds[candidates[0]]
    return ds[var]


def rss_gb():
    return psutil.Process(os.getpid()).memory_info().rss / 1e9


ndvi    = load_da(PROC_DIR / 'NDVI_stack_qa.nc', 'NDVI')
print(f"after NDVI   : RSS = {rss_gb():.2f} GB")

lst     = load_da(PROC_DIR / 'LST_stack_qa.nc',  'LST')
print(f"after LST    : RSS = {rss_gb():.2f} GB")

t2m     = load_da(PROC_DIR / 'T2M_stack.nc',     'T2M')
print(f"after T2M    : RSS = {rss_gb():.2f} GB")

t2m_max = load_da(PROC_DIR / 'T2M_MAX_stack.nc', 'T2M_MAX')
print(f"after T2M_MAX: RSS = {rss_gb():.2f} GB")

t2m_min = load_da(PROC_DIR / 'T2M_MIN_stack.nc', 'T2M_MIN')
print(f"after T2M_MIN: RSS = {rss_gb():.2f} GB")

precip  = load_da(PROC_DIR / 'PRECIP_stack.nc',  'PRECIP')
print(f"after PRECIP : RSS = {rss_gb():.2f} GB")

srtm    = load_da(PROC_DIR / 'SRTM_stack.nc').squeeze(drop=True)
print(f"after SRTM   : RSS = {rss_gb():.2f} GB")

print()
print("ndvi    :", ndvi.shape, ndvi.dims)
print("lst     :", lst.shape)
print("t2m     :", t2m.shape)
print("precip  :", precip.shape)
print("srtm    :", srtm.shape, srtm.dims)
print("years   :", int(ndvi.year.min()), "→", int(ndvi.year.max()))
print("seasons :", sorted(set(str(s) for s in ndvi.season.values)))

In [ ]:
# =============================================================================
# Ra (MJ m-2 d-1) at the AOI centroid — FAO-56 eq. 21.
# =============================================================================

def Ra_mj_m2_day(doy, lat_deg):
    Gsc = 0.0820
    phi = np.deg2rad(lat_deg)
    dr  = 1 + 0.033 * np.cos(2 * np.pi * doy / 365)
    delta = 0.409 * np.sin(2 * np.pi * doy / 365 - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(delta), -1, 1))
    Ra = (24 * 60 / np.pi) * Gsc * dr * (
        ws * np.sin(phi) * np.sin(delta) +
        np.cos(phi) * np.cos(delta) * np.sin(ws)
    )
    return float(Ra)

# Mid-season day-of-year
SEASON_DOY = {'JF': 45, 'MAM': 120, 'JJAS': 227, 'OND': 319}
RA_BY_SEASON = {s: Ra_mj_m2_day(SEASON_DOY[s], LAT_DEG) for s in SEASONS}

for s in SEASONS:
    print(f"Ra {s:5s}: {RA_BY_SEASON[s]:.2f} MJ m-2 d-1")

In [ ]:
# =============================================================================
# PET = 0.0023 * Ra * (Tmean + 17.8) * sqrt(Tmax - Tmin) * n_days
# Memory-safe: one (y, x) slice at a time.
# Frees t2m/t2m_max/t2m_min immediately after.
# =============================================================================

import gc

SEASON_DAYS = {'JF': 59, 'MAM': 92, 'JJAS': 122, 'OND': 92}

def compute_pet(t_mean, t_max, t_min, ra, n_days):
    dtr = np.clip(t_max - t_min, 0.1, None)
    pet_daily = 0.0023 * ra * (t_mean + 17.8) * np.sqrt(dtr)
    return np.clip(pet_daily, 0, None) * n_days

pet_arr = np.full(t2m.shape, np.nan, dtype='float32')

seasons_arr = np.array([str(s) for s in t2m.season.values])

for i, se in enumerate(tqdm(seasons_arr, desc='PET')):
    tmean_slice = t2m.values[i].astype('float32', copy=False)
    tmax_slice  = t2m_max.values[i].astype('float32', copy=False)
    tmin_slice  = t2m_min.values[i].astype('float32', copy=False)
    pet_arr[i] = compute_pet(tmean_slice, tmax_slice, tmin_slice,
                             RA_BY_SEASON[se], SEASON_DAYS[se])

print(f"after PET loop: RSS = {rss_gb():.2f} GB")

pet = xr.DataArray(
    pet_arr,
    dims=('time', 'y', 'x'),
    coords={'time': t2m.time, 'year': t2m.year, 'season': t2m.season,
            'y': t2m.y, 'x': t2m.x},
    name='PET',
    attrs={'units': 'mm/season', 'method': 'Hargreaves (1985)'},
)
pet = pet.rio.write_crs(CRS, inplace=False)
pet.to_netcdf(DERIVED_DIR / 'PET_stack.nc', engine='netcdf4')
print(f"✓ PET saved. Range: {float(np.nanmin(pet_arr)):.1f} – {float(np.nanmax(pet_arr)):.1f} mm/season")

# --- FREE the T2M inputs; not needed anymore ---
del t2m, t2m_max, t2m_min
gc.collect()
print(f"after freeing T2M inputs: RSS = {rss_gb():.2f} GB")

In [ ]:
# =============================================================================
# Water balance D = precipitation − PET.
# Memory-safe: slice-by-slice subtraction.
# =============================================================================

d_arr = np.full(pet_arr.shape, np.nan, dtype='float32')

for i in tqdm(range(pet_arr.shape[0]), desc='D'):
    p_slice = precip.values[i].astype('float32', copy=False)
    d_arr[i] = p_slice - pet_arr[i]

print(f"after D loop: RSS = {rss_gb():.2f} GB")

D = xr.DataArray(
    d_arr,
    dims=('time', 'y', 'x'),
    coords={'time': pet.time, 'year': pet.year, 'season': pet.season,
            'y': pet.y, 'x': pet.x},
    name='D',
    attrs={'units': 'mm/season', 'method': 'P - PET'},
)
D = D.rio.write_crs(CRS, inplace=False)
D.to_netcdf(DERIVED_DIR / 'D_stack.nc', engine='netcdf4')

print(f"✓ D saved. Range: {float(np.nanmin(d_arr)):.1f} – {float(np.nanmax(d_arr)):.1f} mm/season")
print(f"after D save: RSS = {rss_gb():.2f} GB")

# Free PET from memory — D and SPEI don't need it anymore
del pet, pet_arr
gc.collect()
print(f"after del pet: RSS = {rss_gb():.2f} GB")

In [ ]:
# =============================================================================
# SPEI via log-logistic distribution (Vicente-Serrano et al. 2010).
# Numba-accelerated per-pixel loop. Memory-safe: one scale at a time,
# frees the temporary arrays between scales.
# =============================================================================

@njit(cache=True)
def _inv_norm(p):
    a = np.array([-3.969683028665376e+01,  2.209460984245205e+02,
                  -2.759285104469687e+02,  1.383577518672690e+02,
                  -3.066479806614716e+01,  2.506628277459239e+00])
    b = np.array([-5.447609879822406e+01,  1.615858368580409e+02,
                  -1.556989798598866e+02,  6.680131188771972e+01,
                  -1.328068155288572e+01])
    c = np.array([-7.784894002430293e-03, -3.223964580411365e-01,
                  -2.400758277161838e+00, -2.549732539343734e+00,
                   4.374664141464968e+00,  2.938163982698783e+00])
    d = np.array([ 7.784695709041462e-03,  3.224671290700398e-01,
                   2.445134137142996e+00,  3.754408661907416e+00])
    p_low, p_high = 0.02425, 1 - 0.02425

    if p < p_low:
        q = np.sqrt(-2 * np.log(p))
        return (((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5]) / \
                ((((d[0]*q+d[1])*q+d[2])*q+d[3])*q+1)
    elif p <= p_high:
        q = p - 0.5
        r = q * q
        return (((((a[0]*r+a[1])*r+a[2])*r+a[3])*r+a[4])*r+a[5])*q / \
                (((((b[0]*r+b[1])*r+b[2])*r+b[3])*r+b[4])*r+1)
    else:
        q = np.sqrt(-2 * np.log(1 - p))
        return -(((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5]) / \
                 ((((d[0]*q+d[1])*q+d[2])*q+d[3])*q+1)


@njit(parallel=True, cache=True)
def _spei_pixelwise(D_vals, scale, weights, min_valid=20):
    nt, ny, nx = D_vals.shape
    out = np.full((nt, ny, nx), np.nan, dtype=np.float32)

    for j in prange(ny):
        for k in range(nx):
            series = D_vals[:, j, k]
            if np.sum(np.isfinite(series)) < min_valid:
                continue

            acc = np.full(nt, np.nan, dtype=np.float32)
            for i in range(nt):
                lo = max(0, i - scale + 1)
                win = series[lo:i+1]
                ww  = weights[lo:i+1]
                if not np.all(np.isfinite(win)):
                    continue
                acc[i] = np.sum(win * ww) / np.sum(ww) * np.sum(ww)

            valid = acc[np.isfinite(acc)]
            if valid.size < min_valid:
                continue

            shift = -np.min(valid) + 0.01
            x = np.sort(valid + shift)
            n = x.size

            b0 = np.mean(x)
            idx = np.arange(n)
            b1 = np.mean(x * (idx / (n - 1)))
            b2 = np.mean(x * (idx * (idx - 1)) / ((n - 1) * (n - 2)))
            l1 = b0
            l2 = 2 * b1 - b0
            l3 = 6 * b2 - 6 * b1 + b0

            if l2 == 0 or l3 == 0:
                continue
            beta  = 2.0 / (6.0 * (l3 / l2))
            if beta <= 0:
                beta = 2.0
            alpha = l2 * np.sin(np.pi / beta) / (np.pi / beta)
            gamma_ = l1 - alpha * (1.0 / beta) * (np.pi / np.sin(np.pi / beta))

            for i in range(nt):
                if not np.isfinite(acc[i]):
                    continue
                xx = acc[i] + shift
                if xx <= gamma_:
                    continue
                p = 1.0 / (1.0 + (alpha / (xx - gamma_)) ** beta)
                if p < 1e-6: p = 1e-6
                if p > 1 - 1e-6: p = 1 - 1e-6
                out[i, j, k] = _inv_norm(p)

    return out


weights = np.array([SEASON_MONTHS[str(s)] for s in D.season.values],
                   dtype='float32')

D_vals = d_arr  # reuse the array from Cell 5, no copy

for spei_name, scale in SPEI_SCALES.items():
    print(f"\n▶ Computing {spei_name} (scale = {scale} season-steps) ...")
    arr = _spei_pixelwise(D_vals, scale, weights)
    print(f"after {spei_name} compute: RSS = {rss_gb():.2f} GB")

    da = xr.DataArray(
        arr,
        dims=('time', 'y', 'x'),
        coords={'time': D.time, 'year': D.year, 'season': D.season,
                'y': D.y, 'x': D.x},
        name=spei_name,
        attrs={'units': 'standardized', 'method': 'log-logistic SPEI'},
    )
    da = da.rio.write_crs(CRS, inplace=False)
    da.to_netcdf(DERIVED_DIR / f'{spei_name}_stack.nc', engine='netcdf4')

    finite = np.isfinite(arr)
    print(f"  ✓ {spei_name} saved. Valid {100*finite.sum()/finite.size:.1f}%, "
          f"range {np.nanmin(arr):.2f} to {np.nanmax(arr):.2f}")

    del arr, da
    gc.collect()
    print(f"  after free: RSS = {rss_gb():.2f} GB")

In [ ]:
# =============================================================================
# FVC = (NDVI - NDVI_soil) / (NDVI_veg - NDVI_soil)
# Per-season endmembers from 5th/95th percentiles of NDVI.
# Memory-safe: slice-by-slice.
# =============================================================================

fvc_arr = np.full(ndvi.shape, np.nan, dtype='float32')

seasons_arr = np.array([str(s) for s in ndvi.season.values])

for i, se in enumerate(tqdm(seasons_arr, desc='FVC')):
    nd = ndvi.values[i].astype('float32', copy=False)
    finite = np.isfinite(nd)
    if finite.sum() < 100:
        continue

    ndvi_soil = np.nanpercentile(nd, 5)
    ndvi_veg  = np.nanpercentile(nd, 95)
    denom = ndvi_veg - ndvi_soil
    if denom <= 0:
        continue

    fvc_arr[i] = np.clip((nd - ndvi_soil) / denom, 0, 1)

print(f"after FVC loop: RSS = {rss_gb():.2f} GB")

fvc = xr.DataArray(
    fvc_arr,
    dims=('time', 'y', 'x'),
    coords={'time': ndvi.time, 'year': ndvi.year, 'season': ndvi.season,
            'y': ndvi.y, 'x': ndvi.x},
    name='FVC',
    attrs={'units': '0-1', 'method': 'dimidiate pixel model'},
)
fvc = fvc.rio.write_crs(CRS, inplace=False)
fvc.to_netcdf(DERIVED_DIR / 'FVC_stack.nc', engine='netcdf4')

print(f"✓ FVC saved. Range: {float(np.nanmin(fvc_arr)):.3f} – {float(np.nanmax(fvc_arr)):.3f}")

# Free NDVI, LST, precip — no longer needed
del ndvi, lst, precip
gc.collect()
print(f"after free: RSS = {rss_gb():.2f} GB")

In [ ]:
# =============================================================================
# Forest mask — load FOREST_MASK.tif from data/obj1/.
# Values: 1 = tree cover, nodata elsewhere.
# =============================================================================

# --- Get master grid shape and transform from a saved stack ---
with xr.open_dataset(DERIVED_DIR / 'D_stack.nc') as _ds:
    master_shape     = (_ds.sizes['y'], _ds.sizes['x'])
    master_transform = _ds['D'].rio.transform()
print(f"Master grid: shape={master_shape}, transform={master_transform}")

# --- Load forest mask tif ---
mask_path = OBJ1_DIR / 'FOREST_MASK.tif'
print(f"Loading: {mask_path.name}")

fm = rxr.open_rasterio(mask_path, masked=True).squeeze()

if fm.rio.crs is None:
    fm = fm.rio.write_crs(CRS, inplace=False)

# --- Align to master grid if needed ---
if fm.shape != master_shape or str(fm.rio.crs) != CRS:
    print(f"  Reprojecting from {fm.rio.crs} / {fm.shape} to master grid")
    fm = fm.rio.reproject(
        CRS,
        shape=master_shape,
        transform=master_transform,
        resampling=rxr.enums.Resampling.nearest,
    )

fm = (fm == 1).astype('float32')
fm = fm.where(fm == 1)   # 1 inside forest, NaN outside
fm.name = 'FOREST_MASK'
fm.attrs['source'] = 'ESA WorldCover 2021 v200, tree-cover class 10'
fm = fm.rio.write_crs(CRS, inplace=False)
fm.to_netcdf(DERIVED_DIR / 'FOREST_MASK.nc', engine='netcdf4')

frac = float((fm == 1).sum()) / fm.size
print(f"✓ Forest mask saved. Forest covers {frac:.2%} of grid.")

In [ ]:
# =============================================================================
# Forest density = FVC masked to forest.
# Pre-allocated version: builds the full (time, y, x) array in memory,
# writes it in one shot. Peak RAM ~3 GB, but reliable.
# =============================================================================

import gc

# Check RSS before starting
print(f"RSS before Cell 9: {rss_gb():.2f} GB")

# --- Forest mask (already in memory from Cell 8) ---
fm_vals = fm.values.astype('float32')   # (y, x)

# --- Open FVC stack lazily ---
fvc_ds = xr.open_dataset(DERIVED_DIR / 'FVC_stack.nc')
fvc_da = fvc_ds['FVC']

nt = fvc_da.sizes['time']
ny = fvc_da.sizes['y']
nx = fvc_da.sizes['x']
print(f"Allocating ({nt}, {ny}, {nx}) float32 = {nt*ny*nx*4/1e9:.2f} GB")

# --- Pre-allocate output array ---
fd_arr = np.full((nt, ny, nx), np.nan, dtype='float32')

# --- Fill slice-by-slice ---
for i in tqdm(range(nt), desc='Forest density'):
    slice_vals = fvc_da.isel(time=i).values.astype('float32', copy=False)
    fd_arr[i] = slice_vals * fm_vals
    del slice_vals

print(f"after fill: RSS = {rss_gb():.2f} GB")

# --- Wrap with coords and write in one shot ---
forest_density = xr.DataArray(
    fd_arr,
    dims=('time', 'y', 'x'),
    coords={
        'time':   fvc_da.time.values,
        'year':   ('time', fvc_da.year.values),
        'season': ('time', fvc_da.season.values),
        'y':      fvc_da.y.values,
        'x':      fvc_da.x.values,
    },
    name='FOREST_DENSITY',
    attrs={'units': '0-1', 'note': 'FVC masked to ESA WorldCover tree cover'},
)
forest_density = forest_density.rio.write_crs(CRS, inplace=False)
forest_density.to_netcdf(DERIVED_DIR / 'FOREST_DENSITY_stack.nc', engine='netcdf4')

fvc_ds.close()

# --- Verify ---
finite = np.isfinite(fd_arr)
print(f"✓ Forest density saved. Valid {100*finite.sum()/finite.size:.1f}%, "
      f"range {np.nanmin(fd_arr):.3f} – {np.nanmax(fd_arr):.3f}")
print(f"after write: RSS = {rss_gb():.2f} GB")

# --- Free ---
del fd_arr, forest_density, fvc_da
gc.collect()
print(f"after free: RSS = {rss_gb():.2f} GB")

In [ ]:
# =============================================================================
# Summary of everything produced in data/derived/
# =============================================================================

summary = []
for f in sorted(DERIVED_DIR.glob('*.nc')):
    ds = xr.open_dataset(f)
    var = list(ds.data_vars)[0]
    da = ds[var]
    vals = da.values
    finite = np.isfinite(vals)
    summary.append({
        'file':     f.name,
        'variable': var,
        'shape':    str(da.shape),
        'valid_%':  round(100 * finite.sum() / finite.size, 1),
        'min':      float(np.nanmin(vals)) if finite.any() else np.nan,
        'max':      float(np.nanmax(vals)) if finite.any() else np.nan,
    })
    ds.close()

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
summary_df.to_csv(LOG_DIR / 'derived_summary.csv', index=False)

In [ ]:
# =============================================================================
# Export GeoTIFFs for presentation: 6 snapshot years, weighted annual mean
# across the four seasons.
#
# Memory-safe: reads each NetCDF stack lazily, one variable at a time.
# =============================================================================

def annual_weighted_mean(da):
    """Weighted mean across seasons per year. Returns DataArray dims (year, y, x)."""
    w = np.array([SEASON_MONTHS[str(s)] for s in da.season.values],
                 dtype='float32')
    w_b = xr.DataArray(w, dims='time', coords={'time': da.time})

    valid = da.notnull()
    num = (da.fillna(0) * w_b).groupby('year').sum(dim='time')
    den = (valid * w_b).groupby('year').sum(dim='time')
    return num / den.where(den > 0)


TIF_NAMES = ['PET', 'D', 'SPEI_3', 'SPEI_6', 'SPEI_12',
             'FVC', 'FOREST_DENSITY']

print(f"Exporting TIFs for {SNAPSHOT_YEARS}...\n")

for var_name in TIF_NAMES:
    nc_path = DERIVED_DIR / f'{var_name}_stack.nc'
    if not nc_path.exists():
        print(f"⚠ {var_name}: no stack, skipping")
        continue

    ds = xr.open_dataset(nc_path)
    da = ds[var_name]

    try:
        annual = annual_weighted_mean(da)
    except Exception as e:
        print(f"✗ {var_name}: failed annual mean — {e}")
        ds.close()
        continue

    out_folder = TIF_DIR / var_name
    out_folder.mkdir(parents=True, exist_ok=True)

    wrote = 0
    for yr in SNAPSHOT_YEARS:
        if yr not in annual.year.values:
            continue
        snap = annual.sel(year=yr).squeeze()
        if not np.isfinite(snap.values).any():
            print(f"  {var_name} {yr}: all-NaN, skipping")
            continue

        out_tif = out_folder / f'{var_name}_{yr}.tif'
        snap2d = snap.rio.write_crs(CRS, inplace=False)
        snap2d = snap2d.rio.write_nodata(NODATA, inplace=False)
        snap2d.rio.to_raster(
            out_tif, dtype='float32', compress='LZW',
            tiled=True, nodata=NODATA,
        )
        wrote += 1

    print(f"✓ {var_name:15s}: {wrote} tifs → {out_folder.name}/")
    ds.close()

# Forest mask — single static tif
fm_ds = xr.open_dataset(DERIVED_DIR / 'FOREST_MASK.nc')
fm_da = fm_ds['FOREST_MASK'].rio.write_crs(CRS, inplace=False)
fm_da = fm_da.rio.write_nodata(NODATA, inplace=False)
fm_out = TIF_DIR / 'FOREST_MASK'
fm_out.mkdir(parents=True, exist_ok=True)
fm_da.rio.to_raster(
    fm_out / 'FOREST_MASK.tif',
    dtype='float32', compress='LZW', tiled=True, nodata=NODATA,
)
print(f"✓ FOREST_MASK   : 1 tif → FOREST_MASK/")
fm_ds.close()

print(f"\nAll tifs written under: {TIF_DIR}")